# Reproduction of Adult dataset experiments

In this notebook we reproduce the results from Table 2 of the DECAF paper. We compare various methods for generating debiased data using the DECAF model against synthetic data generated using benchmark models GAN, WGAN-GP and FairGAN. As described in the paper we run all experiments (as implemented in this notebook) 10 times and avarage the results.

In [1]:
with open('repNum.txt',"r") as f:
    repNum = f.read().strip()
    repNum = float(repNum)

In [2]:
repNum

9.0

In [3]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.metrics import confusion_matrix


from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

from data import load_adult, preprocess_adult
from metrics import DP, FTU
from train import train_decaf, train_fairgan, train_vanilla_gan, train_wgan_gp


## Loading data

In [4]:
dataset = load_adult()
dataset.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516.0,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,<=50K
1,50,Self-emp-not-inc,83311.0,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,13.0,United-States,<=50K
2,38,Private,215646.0,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,<=50K
3,53,Private,234721.0,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0.0,0.0,40.0,United-States,<=50K
4,28,Private,338409.0,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,40.0,Cuba,<=50K


Preprocess the data next in order to make it suitable for training models on.

In [5]:
dataset = preprocess_adult(dataset)
dataset.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,0.301370,0.833333,0.043350,0.000000,0.800000,0.333333,0.615385,0.6,0.0,1.0,0.02174,0.0,0.397959,0.0,1.0
1,0.452055,0.166667,0.047274,0.000000,0.800000,0.000000,0.307692,0.4,0.0,1.0,0.00000,0.0,0.122449,0.0,1.0
2,0.287671,0.000000,0.136877,0.200000,0.533333,0.166667,0.461538,0.6,0.0,1.0,0.00000,0.0,0.397959,0.0,1.0
3,0.493151,0.000000,0.149792,0.133333,0.400000,0.000000,0.461538,0.4,1.0,1.0,0.00000,0.0,0.397959,0.0,1.0
4,0.150685,0.000000,0.219998,0.000000,0.800000,0.000000,0.384615,0.0,1.0,0.0,0.00000,0.0,0.397959,0.3,1.0


Split the dataset into train and test folds. Test fold size is 2000.

In [6]:
# Split data into train and testing sets
dataset_train, dataset_test = train_test_split(dataset, test_size=2000,
                                               stratify=dataset['income'])

print('Size of train set:', len(dataset_train))
print('Size of test set:', len(dataset_test))

Size of train set: 43222
Size of test set: 2000


### Defining the DAG

We need to define a DAG which captures the biases of the dataset. As described in the DECAF paper normally a causal discovery algorithm is used. In this notebook we simply copy the DAG which as described in the Zhang et al. paper which is the one also used in the DECAF paper.

In [7]:
# Define DAG for Adult dataset
dag = [
    # Edges from race
    ['race', 'occupation'],
    ['race', 'income'],
    ['race', 'hours-per-week'],
    ['race', 'education'],
    ['race', 'marital-status'],

    # Edges from age
    ['age', 'occupation'],
    ['age', 'hours-per-week'],
    ['age', 'income'],
    ['age', 'workclass'],
    ['age', 'marital-status'],
    ['age', 'education'],
    ['age', 'relationship'],
    
    # Edges from sex
    ['sex', 'occupation'],
    ['sex', 'marital-status'],
    ['sex', 'income'],
    ['sex', 'workclass'],
    ['sex', 'education'],
    ['sex', 'relationship'],
    
    # Edges from native country
    ['native-country', 'marital-status'],
    ['native-country', 'hours-per-week'],
    ['native-country', 'education'],
    ['native-country', 'workclass'],
    ['native-country', 'income'],
    ['native-country', 'relationship'],
    
    # Edges from marital status
    ['marital-status', 'occupation'],
    ['marital-status', 'hours-per-week'],
    ['marital-status', 'income'],
    ['marital-status', 'workclass'],
    ['marital-status', 'relationship'],
    ['marital-status', 'education'],
    
    # Edges from education
    ['education', 'occupation'],
    ['education', 'hours-per-week'],
    ['education', 'income'],
    ['education', 'workclass'],
    ['education', 'relationship'],
    
    # All remaining edges
    ['occupation', 'income'],
    ['hours-per-week', 'income'],
    ['workclass', 'income'],
    ['relationship', 'income'],
]

def dag_to_idx(df, dag):
    """Convert columns in a DAG to the corresponding indices."""

    dag_idx = []
    for edge in dag:
        dag_idx.append([df.columns.get_loc(edge[0]), df.columns.get_loc(edge[1])])

    return dag_idx

# Convert the DAG to one that can be provided to the DECAF model
dag_seed = dag_to_idx(dataset, dag)
print(dag_seed)

[[8, 6], [8, 14], [8, 12], [8, 3], [8, 5], [0, 6], [0, 12], [0, 14], [0, 1], [0, 5], [0, 3], [0, 7], [9, 6], [9, 5], [9, 14], [9, 1], [9, 3], [9, 7], [13, 5], [13, 12], [13, 3], [13, 1], [13, 14], [13, 7], [5, 6], [5, 12], [5, 14], [5, 1], [5, 7], [5, 3], [3, 6], [3, 12], [3, 14], [3, 1], [3, 7], [6, 14], [12, 14], [1, 14], [7, 14]]


It's also necessary to define edges we want to remove from the DAG in order to meet the various fairness criteria described in the paper.

In [8]:
def create_bias_dict(df, edge_map):
    """
    Convert the given edge tuples to a bias dict used for generating
    debiased synthetic data.
    """
    bias_dict = {}
    for key, val in edge_map.items():
        bias_dict[df.columns.get_loc(key)] = [df.columns.get_loc(f) for f in val]
    
    return bias_dict

# Bias dictionary to satisfy FTU
bias_dict_ftu = create_bias_dict(dataset, {'income': ['sex']})
print('Bias dict FTU:', bias_dict_ftu)

# Bias dictionary to satisfy DP
bias_dict_dp = create_bias_dict(dataset, {'income': [
    'occupation', 'hours-per-week', 'marital-status', 'education', 'sex',
    'workclass', 'relationship']})
print('Bias dict DP:', bias_dict_dp)

# Bias dictionary to satisfy CF
bias_dict_cf = create_bias_dict(dataset, {'income': [
    'marital-status', 'sex']})
print('Bias dict CF:', bias_dict_cf)

Bias dict FTU: {14: [9]}
Bias dict DP: {14: [6, 12, 5, 3, 9, 1, 7]}
Bias dict CF: {14: [5, 9]}


## Experiments

We have loaded and preprocessed the data and we are ready to run the experiments. For each experiment we train a generative model, sample synthetic data from the trained model and then obtain metrics by training and evaluating a downstream multi-layer perceptron using the test fold we generated in the previous section. We use the MLP model from `sklearn` with default parameters which matches the settings described in Appendix D of the paper.

In [9]:
def eval_model(dataset_train, dataset_test):
    """Helper function that prints evaluation metrics."""

    X_train, y_train = dataset_train.drop(columns=['income']), dataset_train['income']
    X_test, y_test = dataset_test.drop(columns=['income']), dataset_test['income']

    clf = MLPClassifier(max_iter=1000)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    auroc = roc_auc_score(y_test, y_pred)
    dp = DP(clf, X_test)
    ftu = FTU(clf, X_test)
    conf = confusion_matrix(y_test, y_pred)

    return {'precision': precision, 'recall': recall, 'auroc': auroc,
            'dp': dp, 'ftu': ftu, "confMatrix":conf}

Make the result dict:

In [10]:
results = {}

### Original dataset

As a benchmark we want to first train the downstream model on the original dataset.

In [11]:
res = eval_model(dataset_train, dataset_test)
print(res)

{'precision': 0.8813018506700702, 'recall': 0.918218085106383, 'auroc': 0.7716090425531915, 'dp': 0.2041507806514069, 'ftu': 0.02849999999999997, 'confMatrix': array([[ 310,  186],
       [ 123, 1381]])}


In [12]:
results['original'] = res

In the following sections we train various models in order to reproduce the results from Table 2 of the DECAF paper.

### GAN

In [13]:
synth_data = train_vanilla_gan(dataset_train)
synth_data.head()

2023-02-01 17:27:02.309606: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2023-02-01 17:27:02.331950: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
Synthetic data generation: 100%|███████████████████| 338/338 [00:03<00:00, 103.34it/s]


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,0.053421,0.166667,-0.009871,0.000000,0.466667,0.166667,0.615385,0.4,0.75,1.0,0.017775,-0.047426,0.058159,0.350,0.0
1,0.054810,0.500000,-0.004638,0.266667,0.866667,0.000000,1.000000,0.2,0.50,1.0,0.015750,-0.047031,0.055336,0.275,1.0
2,0.053421,0.000000,-0.009871,0.733333,0.000000,0.500000,0.307692,0.6,0.00,0.0,0.017775,-0.047426,0.058159,0.425,1.0
3,0.053421,0.000000,-0.009871,0.866667,0.133333,0.166667,0.692308,0.4,1.00,1.0,0.017775,-0.047426,0.058159,0.250,0.0
4,0.053421,0.166667,-0.009871,0.866667,0.666667,0.500000,0.230769,0.6,0.25,0.0,0.017775,-0.047426,0.058159,0.750,1.0


In [14]:
res = eval_model(synth_data, dataset_test)
print(res)

{'precision': 0.6415525114155252, 'recall': 0.18683510638297873, 'auroc': 0.4351514241592313, 'dp': 0.09083818516326467, 'ftu': 0.094, 'confMatrix': array([[ 339,  157],
       [1223,  281]])}


In [15]:
results['GAN'] = res

### WGAN-GP

In [16]:
synth_data = train_wgan_gp(dataset_train)
synth_data.head()

Synthetic data generation: 100%|██████████████████████| 87/87 [00:01<00:00, 82.01it/s]


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,0.195869,0.000000,0.085711,0.933333,0.333333,0.166667,0.307692,1.0,0.50,1.0,-0.066989,-0.000984,0.319989,0.725,0.0
1,0.192167,0.333333,0.081202,1.000000,0.066667,0.833333,0.076923,1.0,0.25,1.0,-0.070033,-0.001802,0.316902,0.150,0.0
2,0.219452,0.000000,0.093251,0.466667,0.600000,0.666667,0.384615,0.0,0.00,1.0,-0.056479,0.012990,0.345202,0.575,1.0
3,0.215466,0.000000,0.104950,0.400000,0.000000,0.000000,0.076923,0.6,0.75,0.0,-0.071660,0.011329,0.342316,0.825,0.0
4,0.199790,0.166667,0.091932,0.266667,0.200000,0.666667,0.769231,0.2,0.25,1.0,-0.070224,0.002197,0.321959,0.150,1.0


In [17]:
res = eval_model(synth_data, dataset_test)
print(res)

{'precision': 0.8621553884711779, 'recall': 0.22872340425531915, 'auroc': 0.5589181537405629, 'dp': 0.17645749888740542, 'ftu': 0.07450000000000001, 'confMatrix': array([[ 441,   55],
       [1160,  344]])}


In [18]:
results['WGAN-GP'] = res

### FairGAN

In [19]:
synth_data = train_fairgan(dataset_train)
synth_data.head()

INFO:tensorflow:Restoring parameters from cache/fairgan
burning in


2023-02-01 17:27:35.455635: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:196] None of the MLIR optimization passes are enabled (registered 0 passes)


generating


/Users/andrewelliott/Documents/hsbcWork/mahedPaper/FairSyn/UvA_FACT2022-main/env/lib/python3.8/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,0.000000,0.000000,0.000000,0.269772,0.457849,0.000000,0.430427,0.000000,1.183803,0.0,0.0,0.411983,0.000000,0.873224,1.0
1,0.000000,0.427302,0.000000,0.601462,0.000000,0.382515,0.386972,0.000000,1.046633,0.0,0.0,0.427575,0.000000,1.118740,1.0
2,0.462100,0.228834,0.037286,0.691577,0.061365,0.211163,0.228353,0.178641,0.804314,0.0,0.0,0.384807,0.265999,1.006498,1.0
3,0.000000,0.318045,0.000000,0.473922,0.087371,0.106160,0.969486,0.000000,0.959870,0.0,0.0,0.376287,0.000000,0.331646,1.0
4,0.244601,0.019734,0.189285,0.475722,0.071531,0.492422,0.587184,0.000000,0.772654,0.0,0.0,0.418000,0.000000,0.483446,1.0


In [20]:
res = eval_model(synth_data, dataset_test)
print(res)

{'precision': 0.7784743991640544, 'recall': 0.9906914893617021, 'auroc': 0.5679263898421414, 'dp': 0.03301997164604353, 'ftu': 0.08900000000000008, 'confMatrix': array([[  72,  424],
       [  14, 1490]])}


In [21]:
results['FairGAN'] = res

### DECAF

#### DECAF-ND

In [22]:
synth_data = train_decaf(dataset_train, dag_seed)
synth_data.head()

Initialised adjacency matrix as parsed:
 Parameter containing:
tensor([[0., 1., 0., 1., 0., 1., 1., 1., 0., 0., 0., 0., 1., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 1., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 1., 0., 0., 1., 1., 0., 0., 0., 0., 1., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 1.],
        [0., 1., 0., 1., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 1., 0., 1., 0.

/Users/andrewelliott/Documents/hsbcWork/mahedPaper/FairSyn/UvA_FACT2022-main/env/lib/python3.8/site-packages/pytorch_lightning/core/datamodule.py:175: LightningDeprecationWarning: DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.
  rank_zero_deprecation("DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.")
/Users/andrewelliott/Documents/hsbcWork/mahedPaper/FairSyn/UvA_FACT2022-main/env/lib/python3.8/site-packages/pytorch_lightning/core/datamodule.py:170: LightningDeprecationWarning: DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.
  rank_zero_deprecation("DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.")


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
13650,0.138727,0.000414,0.166185,0.542839,0.588004,3.377064e-01,0.076405,0.459632,3.207684e-17,1.0,7.796950e-10,1.085913e-28,0.504219,6.128222e-37,1.0
7201,0.056346,0.004216,0.202817,0.222142,0.696207,3.762265e-01,0.226695,0.253362,0.000000e+00,1.0,5.047459e-15,6.238820e-31,0.386692,7.229552e-28,1.0
791,0.457803,0.000393,0.056647,0.121659,0.561506,8.046934e-06,0.163159,0.409188,1.665047e-38,1.0,7.588803e-08,4.440970e-01,0.390089,6.558839e-30,0.0
12886,0.353013,0.002913,0.073403,0.274819,0.539648,8.229283e-07,0.432652,0.406740,3.096495e-17,1.0,4.692203e-06,2.978182e-29,0.434938,1.760671e-26,0.0
2280,0.538072,0.005629,0.158658,0.268763,0.867950,1.819288e-02,0.521114,0.394797,9.895982e-01,1.0,5.652758e-11,3.915704e-02,0.461485,1.194497e-10,1.0


In [23]:
res = eval_model(synth_data, dataset_test)
print(res)

{'precision': 0.897537728355838, 'recall': 0.7513297872340425, 'auroc': 0.7456245710363761, 'dp': 0.35295856, 'ftu': 0.11749995, 'confMatrix': array([[ 367,  129],
       [ 374, 1130]])}


In [24]:
results['DECAF-ND'] = res

#### DECAF-FTU

In [25]:
synth_data = train_decaf(dataset_train, dag_seed, biased_edges=bias_dict_ftu)
synth_data.head()

Initialised adjacency matrix as parsed:
 Parameter containing:
tensor([[0., 1., 0., 1., 0., 1., 1., 1., 0., 0., 0., 0., 1., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 1., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 1., 0., 0., 1., 1., 0., 0., 0., 0., 1., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 1.],
        [0., 1., 0., 1., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 1., 0., 1., 0.

/Users/andrewelliott/Documents/hsbcWork/mahedPaper/FairSyn/UvA_FACT2022-main/env/lib/python3.8/site-packages/pytorch_lightning/core/datamodule.py:175: LightningDeprecationWarning: DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.
  rank_zero_deprecation("DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.")
/Users/andrewelliott/Documents/hsbcWork/mahedPaper/FairSyn/UvA_FACT2022-main/env/lib/python3.8/site-packages/pytorch_lightning/core/datamodule.py:170: LightningDeprecationWarning: DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.
  rank_zero_deprecation("DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.")


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
13650,0.265700,8.909224e-03,0.160547,0.094256,0.815867,0.392510,0.441238,0.304830,4.026688e-30,0.0,2.825792e-13,6.088622e-14,0.396937,8.690804e-06,0.0
7201,0.063428,1.319451e-05,0.060435,0.141894,0.568487,0.253917,0.496706,0.663678,2.904665e-31,0.0,2.036396e-15,1.859582e-26,0.469708,9.511723e-23,1.0
791,0.533292,6.721929e-07,0.164364,0.358727,0.621188,0.734164,0.661937,0.933893,2.512978e-27,0.0,9.668236e-07,3.060423e-30,0.436838,5.375264e-06,1.0
12886,0.471179,9.391590e-05,0.104556,0.223397,0.617820,0.000907,0.256264,0.400066,5.234417e-28,1.0,3.451162e-10,3.626525e-11,0.450886,9.328502e-22,0.0
2280,0.456967,1.847401e-05,0.084134,0.789699,0.569035,0.000017,0.372464,0.381046,9.059878e-26,1.0,1.058719e-13,9.807576e-27,0.376243,1.012691e-16,1.0


In [26]:
res = eval_model(synth_data, dataset_test)
print(res)

{'precision': 0.8917445482866043, 'recall': 0.7613031914893617, 'auroc': 0.7405306280027453, 'dp': 0.302433, 'ftu': 0.016499996, 'confMatrix': array([[ 357,  139],
       [ 359, 1145]])}


In [27]:
results['DECAF-FTU'] = res

#### DECAF-CF

In [28]:
synth_data = train_decaf(dataset_train, dag_seed, biased_edges=bias_dict_cf)
synth_data.head()

Initialised adjacency matrix as parsed:
 Parameter containing:
tensor([[0., 1., 0., 1., 0., 1., 1., 1., 0., 0., 0., 0., 1., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 1., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 1., 0., 0., 1., 1., 0., 0., 0., 0., 1., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 1.],
        [0., 1., 0., 1., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 1., 0., 1., 0.

/Users/andrewelliott/Documents/hsbcWork/mahedPaper/FairSyn/UvA_FACT2022-main/env/lib/python3.8/site-packages/pytorch_lightning/core/datamodule.py:175: LightningDeprecationWarning: DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.
  rank_zero_deprecation("DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.")
/Users/andrewelliott/Documents/hsbcWork/mahedPaper/FairSyn/UvA_FACT2022-main/env/lib/python3.8/site-packages/pytorch_lightning/core/datamodule.py:170: LightningDeprecationWarning: DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.
  rank_zero_deprecation("DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.")


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
13650,0.301126,0.179602,0.112323,0.181457,0.543904,6.642296e-04,0.312691,0.408491,9.855399e-01,1.0,8.216231e-06,1.388460e-06,0.334952,2.847309e-13,0.0
7201,0.077609,0.235163,0.114201,0.076405,0.559047,2.067127e-02,0.049286,0.380720,4.654683e-27,1.0,8.353685e-09,7.987177e-07,0.453907,4.488004e-14,1.0
791,0.529937,0.000254,0.075931,0.492155,0.558685,1.218893e-04,0.545510,0.065868,3.164342e-11,0.0,2.360412e-17,3.806851e-01,0.368607,2.486419e-21,0.0
12886,0.105300,0.000513,0.121471,0.075922,0.608702,1.043343e-01,0.839007,0.356358,9.884850e-01,1.0,7.217709e-19,4.521991e-24,0.353424,1.996960e-06,1.0
2280,0.538355,0.000020,0.071781,0.890926,0.590208,1.035293e-08,0.648590,0.375756,1.936143e-11,1.0,6.439901e-14,3.028177e-09,0.346176,1.352827e-22,1.0


In [29]:
res = eval_model(synth_data, dataset_test)
print(res)

{'precision': 0.7855887521968365, 'recall': 0.8916223404255319, 'auroc': 0.5768595573095401, 'dp': 0.059765816, 'ftu': 0.035000026, 'confMatrix': array([[ 130,  366],
       [ 163, 1341]])}


In [30]:
results['DECAF-CF'] = res

#### DECAF-DP

In [31]:
synth_data = train_decaf(dataset_train, dag_seed, biased_edges=bias_dict_dp)
synth_data.head()

Initialised adjacency matrix as parsed:
 Parameter containing:
tensor([[0., 1., 0., 1., 0., 1., 1., 1., 0., 0., 0., 0., 1., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 1., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 1., 0., 0., 1., 1., 0., 0., 0., 0., 1., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 1.],
        [0., 1., 0., 1., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 1., 0., 1., 0.

/Users/andrewelliott/Documents/hsbcWork/mahedPaper/FairSyn/UvA_FACT2022-main/env/lib/python3.8/site-packages/pytorch_lightning/core/datamodule.py:175: LightningDeprecationWarning: DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.
  rank_zero_deprecation("DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.")
/Users/andrewelliott/Documents/hsbcWork/mahedPaper/FairSyn/UvA_FACT2022-main/env/lib/python3.8/site-packages/pytorch_lightning/core/datamodule.py:170: LightningDeprecationWarning: DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.
  rank_zero_deprecation("DataModule property `dims` was deprecated in v1.5 and will be removed in v1.7.")


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
13650,0.059728,1.886802e-05,0.198829,0.125943,0.541334,0.373589,0.263073,0.788353,4.172912e-33,0.0,1.414498e-12,1.462779e-27,0.206904,7.366814e-32,1.0
7201,0.085695,1.867841e-04,0.104239,0.091336,0.611505,0.075551,0.133116,0.500039,2.501755e-02,1.0,1.270418e-14,9.972909e-32,0.378195,7.406769e-05,1.0
791,0.042151,8.521603e-08,0.049381,0.054961,0.480557,0.274103,0.117870,0.262343,0.000000e+00,0.0,7.333161e-19,2.480287e-31,0.402779,2.773070e-14,1.0
12886,0.524413,2.569221e-01,0.117481,0.111186,0.589890,0.020757,0.485133,0.410876,1.326932e-33,1.0,7.770917e-05,1.118676e-04,0.483696,4.433695e-18,1.0
2280,0.170831,7.559946e-01,0.108331,0.765798,0.655388,0.005502,0.382305,0.384195,3.293251e-12,1.0,2.205539e-18,4.506612e-13,0.442422,3.967923e-37,0.0


In [32]:
res = eval_model(synth_data, dataset_test)

In [33]:
results['DECAF-DP'] = res

#### SpGan

In [34]:
from spgan import spgan

In [35]:
from importlib import reload

In [36]:
reload(spgan)

<module 'spgan.spgan' from '/Users/andrewelliott/Documents/hsbcWork/mahedPaper/FairSyn/UvA_FACT2022-main/spgan/spgan.py'>

In [37]:
class optC:
    pass

In [38]:
opt = optC()
opt.latent_dim = 40
opt.batch_size = 64
opt.lr = 0.0005
opt.n_epochs = 500
opt.n_critic = 20
opt.clip_value = 0.01

In [39]:
import pandas as pd
def helper(lam):
    opt.lam = lam
    res = spgan.train_spgan(dataset_train.to_numpy(),opt)
    q1 = pd.DataFrame(res)
    q1.columns = dataset_train.columns
    q1['income'] = q1['income']>0.5
    res = eval_model(q1, dataset_test)
    print(res)
    return res

### Run at multiple lambdas

In [40]:
results["spgan-0.0"] = helper(0.0)
results["spgan-0.001"] = helper(0.001)
results["spgan-0.01"] = helper(0.01)
results["spgan-0.025"] = helper(0.025)
results["spgan-0.05"] = helper(0.05)
results["spgan-0.075"] = helper(0.075)
results["spgan-0.1"] = helper(0.1)
results["spgan-1"] = helper(1.0)

[Epoch 0/500] [Batch 0/676] [D loss: -0.055041] [G loss: -0.012863] [d1 loss: 0.696160] [num0: 0.000000] [num1 0.000000]
[Epoch 1/500] [Batch 0/676] [D loss: -0.055726] [G loss: 0.039741] [d1 loss: 0.642716] [num0: 0.000000] [num1 0.000000]
[Epoch 2/500] [Batch 0/676] [D loss: -0.018097] [G loss: 0.031933] [d1 loss: 0.520665] [num0: 0.000000] [num1 0.000000]
[Epoch 3/500] [Batch 0/676] [D loss: -0.009039] [G loss: 0.008349] [d1 loss: 0.588888] [num0: 0.000000] [num1 0.000000]
[Epoch 4/500] [Batch 0/676] [D loss: -0.004486] [G loss: 0.008216] [d1 loss: 0.581450] [num0: 0.000000] [num1 0.000000]
[Epoch 5/500] [Batch 0/676] [D loss: -0.003698] [G loss: 0.005231] [d1 loss: 0.591610] [num0: 0.000000] [num1 4.000000]
[Epoch 6/500] [Batch 0/676] [D loss: -0.002831] [G loss: -0.000305] [d1 loss: 0.562247] [num0: 0.000000] [num1 7.000000]
[Epoch 7/500] [Batch 0/676] [D loss: -0.004547] [G loss: 0.003438] [d1 loss: 0.544889] [num0: 0.000000] [num1 10.000000]
[Epoch 8/500] [Batch 0/676] [D loss: 

### Opt 0.01

## Save data

In [41]:
import pickle

In [42]:
with open("results/data_"+str(repNum)+'.p','wb') as f:
    pickle.dump(results,f)